**Run evidence** · task `06a_create_mas` · run `1071885164067960` · result **SUCCESS** · executed 2026-10-19 17:15 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `06_agent/06a_create_mas`.

# 06a — Agent Bricks Multi-Agent Supervisor (MAS) over the 2 Genie spaces

Creates an **Agent Bricks Multi-Agent Supervisor** that orchestrates the two
domain Genie spaces and exposes them behind a **single serving endpoint**. The
supervisor routes each question to the right space (or both for cross-domain
questions) — this is what the App's chat tab talks to.

* **Agent 1 — Sourcing_Supplier_Risk** → Genie space 1.
* **Agent 2 — Purchase_Order_Analytics** → Genie space 2.

Uses the `/api/2.0/multi-agent-supervisors` API directly (self-contained, no
external package). Idempotent by name.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def _do(method, path, body=None, params=None):
    return w.api_client.do(method, path, body=body, query=params)


## Look up the Genie space IDs created in 05a

In [ ]:
spaces = {r["space_key"]: r["space_id"] for r in spark.table(CFG.fq("_GENIE_SPACES")).collect()}
S1 = spaces["sourcing_supplier_risk"]
S2 = spaces["purchase_order_analytics"]
print(f"Space 1 (sourcing/risk): {S1}")
print(f"Space 2 (PO analytics):  {S2}")

Space 1 (sourcing/risk): 01f1cbe0a7e616b185a50f44a22099be
Space 2 (PO analytics):  01f1cbe0aa081ca0b7f81ab0a72fb650


## Define the two agents (each backed by a Genie space)

In [ ]:
agents = [
    {
        "name": "Sourcing_Supplier_Risk",
        "description": ("Answers sourcing-risk and supply-continuity questions: "
                        "multi-sourcing gaps, single-sourced materials, supplier "
                        "delivery risk, purchasing blocks, quota concentration, and the "
                        "live mitigation cases buyers opened on those risks (open cases, "
                        "risks nobody is working on yet)."),
        "agent_type": "genie",
        "genie_space": {"id": S1},
    },
    {
        "name": "Purchase_Order_Analytics",
        "description": ("Answers purchase-order questions: on-time delivery rates, "
                        "net-price variance across vendors, total spend, and orders "
                        "placed under active supplier delivery risk."),
        "agent_type": "genie",
        "genie_space": {"id": S2},
    },
]

INSTRUCTIONS = (
    "Route sourcing, multi-sourcing, single-sourcing, supplier-risk, purchasing-block "
    "and quota questions to Sourcing_Supplier_Risk. Route on-time delivery, net price, "
    "price variance, spend and orders-under-risk questions to Purchase_Order_Analytics. "
    "For cross-domain questions (e.g. 'did we place POs with single-sourced risky "
    "suppliers?'), consult both agents and combine the answers."
)

## Create or update the MAS (idempotent by name)

In [ ]:

def mas_find_by_name(name):
    # MAS tiles are discovered via the /api/2.0/tiles listing endpoint (filtered to
    # MAS tiles by name), NOT the multi-agent-supervisors endpoint.
    page_token = None
    try:
        while True:
            params = {"filter": f"name_contains={name}&&tile_type=MAS"}
            if page_token:
                params["page_token"] = page_token
            resp = _do("GET", "/api/2.0/tiles", params=params)
            for t in (resp.get("tiles") or []):
                if t.get("name") == name:
                    return t.get("tile_id")
            page_token = resp.get("next_page_token")
            if not page_token:
                break
    except Exception:
        return None
    return None


existing = mas_find_by_name(CFG.mas_name)
if existing:
    print(f"Updating existing MAS '{CFG.mas_name}' ({existing})")
    res = _do("PATCH", f"/api/2.0/multi-agent-supervisors/{existing}",
              body={"tile_id": existing, "name": CFG.mas_name, "agents": agents,
                    "description": "Supervisor over the 2 SAP purchasing Genie spaces.",
                    "instructions": INSTRUCTIONS})
    mas = res.get("multi_agent_supervisor", res)
    tile = mas.get("tile", mas)
    tile_id = tile.get("tile_id", existing)
else:
    res = _do("POST", "/api/2.0/multi-agent-supervisors",
              body={"name": CFG.mas_name, "agents": agents,
                    "description": "Supervisor over the 2 SAP purchasing Genie spaces "
                                   "(Sourcing & Supplier Risk; Purchase Order Analytics).",
                    "instructions": INSTRUCTIONS})
    mas = res["multi_agent_supervisor"]
    tile = mas["tile"]
    tile_id = tile["tile_id"]

endpoint_name = tile["serving_endpoint_name"]
print(f"✓ MAS tile_id={tile_id}")
print(f"✓ Serving endpoint: {endpoint_name}")

Updating existing MAS 'sap_purchasing_supervisor' (5c65408d-f275-451f-896d-0ae3fc69b328)
✓ MAS tile_id=5c65408d-f275-451f-896d-0ae3fc69b328
✓ Serving endpoint: mas-5c65408d-endpoint


## Wait for the serving endpoint to come ONLINE
The MAS provisions its serving endpoint asynchronously (can take several minutes).

In [ ]:

# The authoritative readiness signal is the SERVING ENDPOINT's own state (READY),
# not the MAS tile status field (which reports NOT_APPLICABLE for MAS tiles).
def endpoint_ready():
    try:
        e = w.serving_endpoints.get(endpoint_name)
        return (e.state and str(e.state.ready) == "EndpointStateReady.READY"), str(e.state)
    except Exception as ex:
        return False, f"(not yet created: {ex})"


deadline = time.time() + 1800  # up to 30 min
status = None
while time.time() < deadline:
    ready, status = endpoint_ready()
    print(f"  endpoint state: {status}")
    if ready:
        break
    time.sleep(30)

print(f"Final endpoint state: {status}")

  endpoint state: EndpointState(config_update=<EndpointStateConfigUpdate.NOT_UPDATING: 'NOT_UPDATING'>, ready=<EndpointStateReady.READY: 'READY'>)
Final endpoint state: EndpointState(config_update=<EndpointStateConfigUpdate.NOT_UPDATING: 'NOT_UPDATING'>, ready=<EndpointStateReady.READY: 'READY'>)


## Persist the MAS endpoint name for the App & benchmark

In [ ]:
from pyspark.sql import Row

spark.createDataFrame([Row(
    mas_name=CFG.mas_name, tile_id=tile_id, endpoint_name=endpoint_name,
)]).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CFG.fq("_MAS_ENDPOINT"))

print("Saved to _MAS_ENDPOINT:")
display(spark.table(CFG.fq("_MAS_ENDPOINT")))

Saved to _MAS_ENDPOINT:


mas_name,tile_id,endpoint_name
sap_purchasing_supervisor,5c65408d-f275-451f-896d-0ae3fc69b328,mas-5c65408d-endpoint


## (Optional) Smoke-test the endpoint with a cross-domain question

In [ ]:
# The MAS endpoint uses the OpenAI-style **Responses API**: send an `input` list of
# messages, receive an `output` list of items (function_call + assistant messages).
import requests

try:
    headers = w.config.authenticate()
    url = f"{w.config.host}/serving-endpoints/{endpoint_name}/invocations"
    r = requests.post(url, headers=headers,
                      json={"input": [{"role": "user",
                                       "content": "Which materials have no multi-sourcing strategy?"}]},
                      timeout=160)
    print("HTTP", r.status_code)
    out = r.json().get("output", [])
    # The final assistant message holds the answer text.
    for item in out:
        if item.get("type") == "message":
            for c in item.get("content", []):
                if c.get("type") == "output_text":
                    print("Answer:", c["text"][:500])
except Exception as e:
    print(f"(endpoint query skipped/failed — it may still be warming up: {e})")

HTTP 200
Answer: I'll query the sourcing risk agent to find materials that lack a multi-sourcing strategy.
Answer: <name>Sourcing_Supplier_Risk</name>
Answer: ||MATNR|WERKS|valid_vendor_count|
|-|-|-|-|
|0|MAT100000|1010|1|
|1|MAT100002|1010|1|
|2|MAT100002|1020|1|
|3|MAT100004|1030|1|
|4|MAT100005|1010|1|
|5|MAT100007|1030|1|
|6|MAT100007|1020|1|
|7|MAT100007|1010|1|
|8|MAT100008|1010|1|
|9|MAT100008|1020|1|
|10|MAT100014|1030|1|
|11|MAT100014|1010|1|
|12|MAT100014|1020|1|
|13|MAT100020|1020|1|
|14|MAT100021|1030|1|
|15|MAT100023|1020|1|
|16|MAT100025|1010|1|
|17|MAT100027|1020|1|
|18|MAT100027|1030|1|
|19|MAT100027|1010|1|
|20|MAT100028|1030|1|
|21|
Answer: <name>sap_purchasing_supervisor</name>
Answer: The system has identified **45 material-plant combinations** that lack a multi-sourcing strategy. These are materials that have only one valid vendor (valid_vendor_count = 1) at specific plants, creating single-source dependencies.

Here are the materials without multi-sourcing:

**Ma

In [ ]:
dbutils.notebook.exit(f"06a_create_mas: SUCCESS — endpoint={endpoint_name}, status={status}")

Notebook exited: 06a_create_mas: SUCCESS — endpoint=mas-5c65408d-endpoint, status=EndpointState(config_update=<EndpointStateConfigUpdate.NOT_UPDATING: 'NOT_UPDATING'>, ready=<EndpointStateReady.READY: 'READY'>)
